In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_295K_278464_invert_cb_rot_17O_opt_magres_new.magres') #latest magres file from 2025

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[1.85281308e+02 2.14711323e+00 2.97631917e+00]
 [8.06879096e-02 1.76571023e+02 5.39651829e+00]
 [3.28012261e+00 5.80564094e+00 1.90494290e+02]]

14N2 sigma:
 [[ 1.85281308e+02 -2.14711323e+00 -2.97631917e+00]
 [-8.06879096e-02  1.76571023e+02  5.39651829e+00]
 [-3.28012261e+00  5.80564094e+00  1.90494290e+02]]

14N3 sigma:
 [[ 1.85281308e+02  2.14711323e+00 -2.97631917e+00]
 [ 8.06879096e-02  1.76571023e+02 -5.39651829e+00]
 [-3.28012261e+00 -5.80564094e+00  1.90494290e+02]]

14N4 sigma:
 [[ 1.85281308e+02 -2.14711323e+00  2.97631917e+00]
 [-8.06879096e-02  1.76571023e+02 -5.39651829e+00]
 [ 3.28012261e+00 -5.80564094e+00  1.90494290e+02]]



In [6]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 1.3977979295675185

14N2 sigma:
 1.3977979295675014

14N3 sigma:
 1.3977979295675318

14N4 sigma:
 1.3977979295675191



In [7]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))  # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 0
CS_total[:,:] = atoms.species('N').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('N')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.0204 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.386 -0.261  1.117]
 [-0.261 -0.349 -0.285]
 [ 1.117 -0.285 -0.037]]

CS Tensor:
 [[1.85281e+02 2.14700e+00 2.97600e+00]
 [8.10000e-02 1.76571e+02 5.39700e+00]
 [3.28000e+00 5.80600e+00 1.90494e+02]]

CS isotropic Tensor:
 [[184.116   0.      0.   ]
 [  0.    184.116   0.   ]
 [  0.      0.    184.116]]

CS symmetric Tensor:
 [[185.281   1.114   3.128]
 [  1.114 176.571   5.601]
 [  3.128   5.601 190.494]]

CS antisymmetric Tensor:
 [[ 0.     1.033 -0.152]
 [-1.033  0.    -0.205]
 [ 0.152  0.205  0.   ]]


In [8]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 1.39506203 -0.9672803  -0.42778173] 

 Unsorted Eigenvectors:
 [[-0.74899572  0.62264045  0.22654864]
 [ 0.21448347 -0.09566545  0.97203136]
 [-0.62689892 -0.77663826  0.06189307]] 

Sorted Eigenvalues: 
 [-0.42778173 -0.9672803   1.39506203] 

Sorted Eigenvectors: 
 [[ 0.22654864  0.62264045 -0.74899572]
 [ 0.97203136 -0.09566545  0.21448347]
 [ 0.06189307 -0.77663826 -0.62689892]] 


For CS tensor
 Unsorted Eigenvalues:
 [193.7671681  183.98193163 174.59752095] 

 Unsorted Eigenvectors:
 [[-0.36440444 -0.93124014  0.00109619]
 [-0.30966473  0.12006489 -0.94323495]
 [-0.87824664  0.34405846  0.33212441]] 

Sorted Eigenvalues: 
 [183.98193163 174.59752095 193.7671681 ] 

Sorted Eigenvectors: 
 [[-0.93124014  0.00109619 -0.36440444]
 [ 0.12006489 -0.94323495 -0.30966473]
 [ 0.34405846  0.33212441 -0.87824664]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.4277817299947517 -0.9672803049094324 1.3950620349042073
CSA Tensor Components δyy, δxx, δzz: 
 183.9819316268309 174.59752095220156 193.76716810317424


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        1.39506
etaq            0.38672
iso_cs (ppm)  184.116
csa (ppm)       9.65163
etas            0.972314


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.62264045  0.22654864 -0.74899572]
 [-0.09566545  0.97203136  0.21448347]
 [-0.77663826  0.06189307 -0.62689892]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-4.556475095487899 128.8216992536096 15.979669576475422 

Direction cosine csa: 

[[ 0.00109619 -0.93124014 -0.36440444]
 [-0.94323495  0.12006489 -0.30966473]
 [ 0.33212441  0.34405846 -0.87824664]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
46.01111632453674 151.43157423485357 -40.35732093558005 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -50.97776194139108 chi: 40.791563033016985 xi: 42.09062955448148 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[185.28130774   1.11390057   3.12822089]
 [  1.11390057 176.57102321   5.60107961]
 [  3.12822089   5.60107961 190.49428974]]
CSA Tensor in Tenon Frame: 
 [[175.73273759   1.58123924  -4.22345966]
 [  1.58123924 185.59786346  -3.39141021]
 [ -4.22345966  -3.39141021 191.01601963]]
Quad Tensor in Crystal Frame: 
 [[ 0.38567033 -0.26069942  1.11678984]
 [-0.26069942 -0.34886258 -0.28518195]
 [ 1.11678984 -0.28518195 -0.03680775]]
Quad Tensor in Tenon Frame: 
 [[-0.3608377   0.08522605  0.343451  ]
 [ 0.08522605 -0.93038343  0.27505742]
 [ 0.343451    0.27505742  1.29122113]]
